# FID vs MLflow metrics

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

from gen_cats.evaluation.experiment_artifacts import (
    join_fid_checkpoints,
    join_fid_mlflow_metrics,
    load_mlflow_runs,
)
from gen_cats.evaluation.report_analysis import ensure_plots_dir, write_stats_csv

sns.set_theme(style="whitegrid", context="talk")
PLOTS = ensure_plots_dir("results")
ml = load_mlflow_runs()
joined = join_fid_checkpoints()
compare = join_fid_mlflow_metrics(mlflow_df=ml)
compare.head()

,model,seed,fid,slug,mlflow_status,best_metric,final_epoch,run_uuid
0,beta_vae,0,287.403875,0a2cb55b6c44,FINISHED,0.175847,109.0,0a273cc7d9424fa09cf15f47f0bf6cbe
1,beta_vae,0,287.234212,4a2131d553db,FINISHED,0.152588,36.0,2c3f597c21b946f1a96e92f0ef7c2905
2,beta_vae,42,335.615580,6b9f922c926d,FINISHED,0.293350,40.0,fbc19b093b844841888d12bfbda8f1f8
3,beta_vae,42,310.473785,6e0cd1bd7e7c,FINISHED,0.215415,35.0,0aac1878abb048d38e807ef72f8c4b4d
4,beta_vae,0,310.966543,79d5347123d6,FINISHED,0.213179,43.0,04937dbeb9534ecb8698a03cd3ea235d


In [2]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = compare.dropna(subset=["best_metric"])
sns.scatterplot(
    data=plot_df,
    x="best_metric",
    y="fid",
    hue="model",
    ax=ax,
    s=70,
)
ax.set_xlabel("MLflow best_metric")
ax.set_ylabel("FID")
ax.legend(title="")
fig.tight_layout()
out = PLOTS / "fid_vs_best_metric.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.close(fig)
write_stats_csv(compare, "fid_mlflow_join.csv")
coverage = joined.groupby("model", as_index=False).agg(
    fid_evals=("fid", "count"),
    with_samples_best=("has_samples_best", "sum"),
    with_epoch_grids=("n_epoch_samples", "sum"),
)
write_stats_csv(coverage, "checkpoint_coverage.csv")
plot_df.describe()

,seed,fid,best_metric,final_epoch
count,87.000000,87.000000,87.000000,87.000000
mean,1149.666667,282.925088,-24.405739,83.609195
std,1605.521604,37.033205,41.296411,60.332674
min,0.000000,174.402621,-137.518141,35.000000
25%,0.000000,259.264439,-57.963451,40.000000
50%,42.000000,282.499894,-0.675984,51.000000
75%,3407.000000,306.179593,0.104283,137.500000
max,3407.000000,372.247619,2.934962,235.000000
